In [1]:
!pip install pytesseract pillow scikit-learn joblib matplotlib

In [2]:
import os
import random
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# Create folder structure
for cls in ['invoices', 'receipts', 'contracts']:
    os.makedirs(f'/kaggle/working/training_data/{cls}', exist_ok=True)

# Helper: generate random text based on document type
def generate_text(doc_type):
    if doc_type == 'invoices':
        return f"""INVOICE # {random.randint(1000,9999)}
Date: 2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}
Due Date: 2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}
Amount: ${random.randint(50,5000)}.00
Vendor: Company XYZ
Customer: John Doe"""
    elif doc_type == 'receipts':
        return f"""RECEIPT
Store: {random.choice(['Walmart', 'Target', 'Best Buy'])}
Date: 2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}
Items: {random.randint(1,5)} @ ${random.randint(5,100)} each
Total: ${random.randint(10,500)}.00
Payment: Credit Card"""
    else:  # contracts
        return f"""CONTRACT AGREEMENT
Between: Client A and Provider B
Effective Date: 2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}
Term: {random.randint(12,60)} months
Total Value: ${random.randint(10000,100000)}.00
Signed by both parties."""

# Generate 15 images per class (10+ for good split)
for cls in ['invoices', 'receipts', 'contracts']:
    for i in range(15):
        img = Image.new('RGB', (800, 600), color='white')
        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf", 20)
        except:
            font = ImageFont.load_default()
        text = generate_text(cls)
        draw.text((50,50), text, fill='black', font=font)
        img.save(f'/kaggle/working/training_data/{cls}/{cls}_{i}.png')
print("Synthetic dataset created.")

Synthetic dataset created.


In [3]:
import os
import pytesseract
from PIL import Image

def load_documents(data_dir):
    documents = []
    labels = []
    for doc_type in os.listdir(data_dir):
        folder_path = os.path.join(data_dir, doc_type)
        if not os.path.isdir(folder_path):
            continue
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png','.jpg','.jpeg')):
                file_path = os.path.join(folder_path, filename)
                img = Image.open(file_path)
                # OCR with basic config
                text = pytesseract.image_to_string(img)
                documents.append(text)
                labels.append(doc_type)
    return documents, labels

data_dir = '/kaggle/working/training_data'
documents, labels = load_documents(data_dir)
print(f"Loaded {len(documents)} documents")
print(f"Classes: {set(labels)}")

Loaded 45 documents
Classes: {'contracts', 'invoices', 'receipts'}


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42, stratify=labels
)

# TF-IDF vectorizer (unigrams + bigrams, max 1000 features)
vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words='english',
    ngram_range=(1,2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train classifier
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_vec, y_train)

# Evaluate
y_pred = classifier.predict(X_test_vec)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 100.00%

Classification Report:
              precision    recall  f1-score   support

   contracts       1.00      1.00      1.00         3
    invoices       1.00      1.00      1.00         3
    receipts       1.00      1.00      1.00         3

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9



In [5]:
import joblib

joblib.dump(vectorizer, 'vectorizer.pkl')
joblib.dump(classifier, 'classifier.pkl')
print("Models saved as 'vectorizer.pkl' and 'classifier.pkl'")

Models saved as 'vectorizer.pkl' and 'classifier.pkl'
